# Notebook 11 — AEGIS Binary IDS Baseline Training and Evaluation

Train a **Random Forest** baseline on the **five live-target CICIDS2017 flow features** from Notebook 10. Evaluate on validation first, then **once** on the held-out test set. Save a versioned model artifact, machine-readable metadata, metrics, and charts.

**Important:** These are historical CICIDS2017 results, not evidence of live detection accuracy. Notebook 10 grouped exact five-feature matches to limit leakage; related capture sessions can still leak statistical patterns. Validate live feature units and calculation methods before deployment.

Run all cells in order. Outputs go to `ml/models/cicids2017_baseline/` and `ml/evaluation/cicids2017_baseline/`.

In [1]:
from pathlib import Path
import json, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, accuracy_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score, PrecisionRecallDisplay,
    RocCurveDisplay)
import joblib

# Detect root whether notebook runs from repo or ml/notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'ml' / 'data' / 'processed').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot find repo root with ml/data/processed. Open VS Code in repository.')
SPLIT_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed' / 'model_ready'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'models' / 'cicids2017_baseline'
EVAL_DIR = PROJECT_ROOT / 'ml' / 'evaluation' / 'cicids2017_baseline'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Flow Bytes/s','Flow Packets/s']
TARGET = 'BinaryLabel'
RANDOM_STATE = 42
print('Repo:',PROJECT_ROOT)
print('Split folder:',SPLIT_DIR)
print('Versions:',{'python':sys.version.split()[0], 'scikit-learn':sklearn.__version__, 'pandas':pd.__version__})

Repo: c:\Users\SOYUZ\Documents\CPSC-491\AI-Intrusion-Detection-System
Split folder: c:\Users\SOYUZ\Documents\CPSC-491\AI-Intrusion-Detection-System\ml\data\processed\model_ready
Versions: {'python': '3.11.0', 'scikit-learn': '1.9.0', 'pandas': '3.0.6'}


## 1. Load splits and verify schema

In [2]:
paths={split:SPLIT_DIR/f'CICIDS2017_Binary_{split}.csv' for split in ('train','validation','test')}
for split,p in paths.items():
    if not p.is_file(): raise FileNotFoundError(f'Missing {split} CSV: {p}')
# Read only what the model requires: prevents metadata and labels leaking into the features.
data={split:pd.read_csv(p,usecols=FEATURES+[TARGET]) for split,p in paths.items()}
for split,df in data.items():
    if df.empty: raise ValueError(f'{split} empty')
    if df[FEATURES+[TARGET]].isna().any().any(): raise ValueError(f'{split}: missing values')
    X=df[FEATURES].to_numpy(dtype=np.float64)
    if not np.isfinite(X).all(): raise ValueError(f'{split}: nonfinite features')
    if not set(df[TARGET].unique()).issubset({0,1}): raise ValueError(f'{split}: target not binary')
    print(split, len(df), 'benign:',int((df[TARGET]==0).sum()),'attack:',int((df[TARGET]==1).sum()))
X_train=data['train'][FEATURES].astype('float32')
y_train=data['train'][TARGET].astype('int8')
X_val=data['validation'][FEATURES].astype('float32')
y_val=data['validation'][TARGET].astype('int8')
X_test=data['test'][FEATURES].astype('float32')
y_test=data['test'][TARGET].astype('int8')
print('Ready. Feature order:', FEATURES)

train 1791100 benign: 1490344 attack: 300756
validation 376404 benign: 318106 attack: 58298
test 405023 benign: 338336 attack: 66687
Ready. Feature order: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s']


## 2. Train a conservative baseline

Random Forest does not require standardization. `class_weight="balanced_subsample"` compensates for the imbalance **during training only**. Use modest tree depth and leaf size to limit overfitting and memory usage. This can still use substantial memory on 1.79 million records.

In [3]:
model=RandomForestClassifier(
    n_estimators=100, max_depth=18, min_samples_leaf=10,
    max_features='sqrt', class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_STATE
)
model.fit(X_train,y_train)
print('Training complete! Trees:',len(model.estimators_))

Training complete! Trees: 100


## 3. Evaluate validation data and choose alert threshold

In [4]:
val_prob=model.predict_proba(X_val)[:,list(model.classes_).index(1)]
thresholds=[0.30,0.40,0.50,0.60,0.70,0.80,0.90]
rows=[]
for threshold in thresholds:
    pred=(val_prob >= threshold).astype('int8')
    tn,fp,fn,tp=confusion_matrix(y_val,pred,labels=[0,1]).ravel()
    p,r,f,_=precision_recall_fscore_support(y_val,pred,average='binary',zero_division=0)
    rows.append({'threshold':threshold,'precision':p,'recall':r,'f1':f,
                 'false_positive_rate':fp/(fp+tn) if fp+tn else float('nan'),
                 'false_positives':int(fp),'false_negatives':int(fn)})
threshold_results=pd.DataFrame(rows)
display(threshold_results.round(5))
threshold_results.to_csv(EVAL_DIR/'validation_thresholds.csv',index=False)
# This policy maximizes validation binary F1 among a predefined small grid.
# Alternative use cases may prioritize lower false-positive rates.
chosen_threshold=float(threshold_results.loc[threshold_results['f1'].idxmax(),'threshold'])
print('Selected threshold from validation F1:',chosen_threshold)
print('Validation ROC AUC:',roc_auc_score(y_val,val_prob))
print('Validation average precision:',average_precision_score(y_val,val_prob))

,threshold,precision,recall,f1,false_positive_rate,false_positives,false_negatives
0,0.3,0.90137,0.99245,0.94472,0.01990,6331,440
1,0.4,0.96017,0.98533,0.97259,0.00749,2383,855
2,0.5,0.96660,0.98401,0.97523,0.00623,1982,932
3,0.6,0.96983,0.97333,0.97158,0.00555,1765,1555
4,0.7,0.97265,0.97103,0.97184,0.00500,1592,1689
5,0.8,0.98337,0.96638,0.97480,0.00300,953,1960
6,0.9,0.98636,0.94427,0.96486,0.00239,761,3249


Selected threshold from validation F1: 0.5
Validation ROC AUC: 0.9987393700935748
Validation average precision: 0.9938411145509706


## 4. Final held-out test evaluation

Do **not** change the threshold after inspecting these test results. If you revise the model, reserve a new independent evaluation set.

In [ ]:
test_prob=model.predict_proba(X_test)[:,list(model.classes_).index(1)]
test_pred=(test_prob>=chosen_threshold).astype('int8')
tn,fp,fn,tp=confusion_matrix(y_test,test_pred,labels=[0,1]).ravel()
test_metrics={
 'threshold':chosen_threshold,
 'accuracy':float(accuracy_score(y_test,test_pred)),
 'balanced_accuracy':float(balanced_accuracy_score(y_test,test_pred)),
 'roc_auc':float(roc_auc_score(y_test,test_prob)),
 'average_precision':float(average_precision_score(y_test,test_prob)),
 'false_positive_rate':float(fp/(fp+tn)),
 'false_negative_rate':float(fn/(fn+tp)),
 'true_negative':int(tn),'false_positive':int(fp),
 'false_negative':int(fn),'true_positive':int(tp),
 'test_samples':len(y_test)
}
print('Held-out test metrics:')
print(json.dumps(test_metrics,indent=2))
print('Classification report:')
print(classification_report(y_test,test_pred,labels=[0,1],target_names=['BENIGN','ATTACK'],zero_division=0))
with (EVAL_DIR/'test_metrics.json').open('w') as f: json.dump(test_metrics,f,indent=2)
with (EVAL_DIR/'test_classification_report.txt').open('w') as f:
    f.write(classification_report(y_test,test_pred,labels=[0,1],target_names=['BENIGN','ATTACK'],zero_division=0))

SyntaxError: unterminated string literal (detected at line 18) (606673821.py, line 18)

## 5. Confusion matrix, ROC curve, precision–recall curve

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test,test_pred,labels=[0,1],display_labels=['BENIGN','ATTACK'],ax=ax,values_format='d',cmap='Blues')
ax.set_title('Held-out test confusion matrix')
fig.tight_layout(); fig.savefig(EVAL_DIR/'test_confusion_matrix.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(6,5))
RocCurveDisplay.from_predictions(y_test,test_prob,ax=ax)
ax.set_title('Held-out test ROC');fig.tight_layout();fig.savefig(EVAL_DIR/'test_roc.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(6,5))
PrecisionRecallDisplay.from_predictions(y_test,test_prob,ax=ax)
ax.set_title('Held-out test precision–recall');fig.tight_layout();fig.savefig(EVAL_DIR/'test_precision_recall.png',dpi=160);plt.show()

## 6. Feature importance (descriptive, not causal)

In [ ]:
importance=pd.DataFrame({'feature':FEATURES,'impurity_importance':model.feature_importances_}).sort_values('impurity_importance',ascending=False)
display(importance)
importance.to_csv(EVAL_DIR/'feature_importance.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4))
ax.barh(importance['feature'][::-1],importance['impurity_importance'][::-1]);ax.set_xlabel('Random Forest impurity importance');fig.tight_layout();fig.savefig(EVAL_DIR/'feature_importance.png',dpi=160);plt.show()

## 7. Save model and inference metadata

Treat model files like executable code: **only load trusted artifacts**. The metadata records feature order, label mapping, decision threshold, and exact training-library version. Confirm live TShark aggregation produces matching values before plugging the model into FastAPI.

In [ ]:
artifact_path=MODEL_DIR/'aegis_cicids2017_binary_rf_v1.joblib'
joblib.dump(model,artifact_path,compress=3)
metadata={
 'model_id':'aegis_cicids2017_binary_rf_v1',
 'algorithm':'RandomForestClassifier',
 'task':'binary_network_flow_classification',
 'feature_names_ordered':FEATURES,
 'label_mapping':{'0':'BENIGN','1':'ATTACK'},
 'positive_class':1,
 'threshold':chosen_threshold,
 'flow_duration_unit_expected':'microseconds',
 'live_feature_equivalence_verified':False,
 'data_source':'CICIDS2017 cleaned flow CSV; notebook 10 grouped five-feature splits',
 'random_state':RANDOM_STATE,
 'versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__,
             'pandas':pd.__version__,'numpy':np.__version__},
 'training_records':len(y_train),
 'validation_records':len(y_val),
 'test_records':len(y_test)
}
with (MODEL_DIR/'aegis_cicids2017_binary_rf_v1.metadata.json').open('w') as f:
    json.dump(metadata,f,indent=2)
print('Saved model:',artifact_path)
print('Saved metadata:',MODEL_DIR/'aegis_cicids2017_binary_rf_v1.metadata.json')
print('Saved evaluation:',EVAL_DIR)
print('NOTE: Model is a baseline, not approved for production live alerts.')

## 8. Optional smoke test for artifact readability

Only loads the model we just created locally. Model persistence via `joblib` is unsafe for untrusted files.

In [ ]:
reloaded=joblib.load(artifact_path)
smoke_prob=reloaded.predict_proba(X_val.iloc[:5])
assert smoke_prob.shape==(5,2)
print('Reload successful. First five validation probabilities:',smoke_prob.tolist())
print('NOTEBOOK 11 COMPLETE — baseline trained, evaluated, and saved.')